# LLM relevance judge for ESCI, on a free Colab T4

Part of [bilingual-product-search](../README.md). Costs nothing: free Colab GPU, an open-weights model, no paid API.

**Question.** Can an open instruction LLM replace the human ESCI labels? Two tests:
1. *Judge.* Label held-out (query, product) pairs as Exact / Substitute / Complement / Irrelevant and compare
   with the human labels (accuracy, Cohen's kappa, confusion matrix), and rerank each query's judged products by
   the LLM's expected gain (NDCG@10), against E5 and the cross-encoders on the same queries.
2. *Distillation.* Label training pairs with the LLM, train the cross-encoder on those labels, and compare it
   with the same cross-encoder trained on the human labels of exactly the same pairs.

**Model.** `Qwen/Qwen2.5-7B-Instruct` (Apache-2.0; reads Japanese), 4-bit NF4 via bitsandbytes, about 5.5 GB of
the T4's 15 GB. (Qwen2.5-3B would be faster but is under the non-commercial Qwen Research License.) The prompt
is `bps.judge.SYSTEM` with the ESCI label definitions plus the query and the product text (title | brand |
colour | bullets, first 400 characters). There is no generation: one forward pass per pair, and the next-token
probabilities of the letters E, S, C, I (renormalised over the four) are the judgement. The system prompt's
keys/values are computed once and shared by every batch (9 pairs/s on a T4 instead of 5; same argmax label on
all 609 pairs checked, probabilities within 0.04). Deterministic, no prompt tuning on the evaluation pairs.

**Pairs** (`scripts/make_judge_data.py`, rebuilt here from the public ESCI files and checked against the
laptop's fingerprint):
- `view`: every judged product of the first 200 test queries per language (10,226 pairs), the same queries as
  the judged-candidate rows of `outputs/search/results*.json`.
- `distill`: a pool of 2,000 training queries per language (train split minus the eval dev queries; no test
  query or test query text, asserted). Labelled in a seeded random query order, alternating languages, so
  whatever finishes within the time budget is a random sample.

**Runs** (free Colab sessions are limited, so each step has a time budget and resumes from its checkpoint):
labels are written in chunks of 20 queries; finished chunks are skipped on a rerun. Keep Google Drive mounted:
a free Colab VM can be recycled without warning (it happened once during this experiment and the labels on the VM
were lost), while chunks on Drive survive and the rerun skips them.

**After** (laptop): unzip `llm_judge.zip` into the repo, then `scripts/judge_report.py` and `run_search.py
--reranker` (see the last cell).

In [ ]:
# same transformers / sentence-transformers as the laptop, so the trained cross-encoders load there unchanged
!pip -q install transformers==4.46.3 sentence-transformers==3.3.1 bitsandbytes==0.50.2

In [ ]:
import os
from pathlib import Path

os.environ["USE_TF"] = "0"   # transformers would otherwise import Colab's TensorFlow

REPO = Path("/content/bilingual-product-search")
if not REPO.exists():
    !git clone -q https://github.com/Praruj/bilingual-product-search {REPO}
os.chdir(REPO)

if os.environ.get("BPS_BASE"):       # e.g. a VM folder when Drive is not wanted; lost on disconnect
    BASE = Path(os.environ["BPS_BASE"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/bps/llm_judge")
LABELS, MODELS = BASE / "labels", BASE / "models"
for p in (LABELS, MODELS):
    p.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data

The ESCI files (Apache-2.0, about 1.2 GB) come from the official GitHub repository. The fingerprint is the one
`make_judge_data.py` printed on the laptop, so both sides label exactly the same pairs and text.

In [ ]:
RAW = Path("/content/esci")
URL = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset"
for f in ("shopping_queries_dataset_examples.parquet", "shopping_queries_dataset_products.parquet"):
    if not (RAW / f).exists():
        !wget -q -P {RAW} {URL}/{f}
PAIRS = Path("/content/judge_pairs.parquet")
!python scripts/make_judge_data.py --raw {RAW} --out {PAIRS}
import json
fp = json.loads(PAIRS.with_suffix(".stats.json").read_text())["fingerprint"]
assert fp == "68262ebd0a8d4e22", f"pairs differ from the laptop's: {fp}"

## 1. Judge the held-out pairs (about 10k pairs)

About 20 minutes. The first call labels one chunk and prints the speed; the next continues from there.

In [ ]:
!python scripts/llm_judge.py label --pairs {PAIRS} --out {LABELS} --set view --max-chunks 1

In [ ]:
!python scripts/llm_judge.py label --pairs {PAIRS} --out {LABELS} --set view --hours 1.5

## 2. Label training pairs for distillation

About 40k pairs (about 1,600 queries) in 1.3 hours. Rerun the cell (in a new session if needed) to label more; the
cross-encoders below are trained on whichever queries are fully labelled at that point.

In [ ]:
!python scripts/llm_judge.py label --pairs {PAIRS} --out {LABELS} --set distill --hours 1.3

## 3. Train the cross-encoder on LLM labels, and on human labels of the same pairs

Same base model, loss (BCE with ESCI gains as targets) and hyperparameters as `colab_finetune.ipynb`. The two
runs differ only in the targets: the gain of the LLM's argmax label vs the human label's gain.

In [ ]:
for target in ("llm", "human"):
    !python scripts/llm_judge.py train --pairs {PAIRS} --labels {LABELS} --target {target} --out {MODELS}/mmarco-minilm-{target}-labels

## Download

One zip with the labels and both models (fp16, about 240 MB each). In the in-app browser `files.download` may
not save; then download `MyDrive/bps/llm_judge.zip` from Google Drive. Check the sha256 on the laptop.

In [ ]:
!cd {BASE} && rm -f ../llm_judge.zip && zip -qr ../llm_judge.zip labels models && sha256sum ../llm_judge.zip && ls -la ../llm_judge.zip

## Back on the laptop

```bash
# unzip llm_judge.zip so that labels/ is data/judge/labels/ and models/ is models/llm_judge/
python scripts/judge_report.py           # agreement + NDCG of LLM scores -> outputs/llm_judge/
python scripts/run_search.py --reranker models/llm_judge/mmarco-minilm-llm-labels --tag llm_distilled
python scripts/run_search.py --reranker models/llm_judge/mmarco-minilm-human-labels --tag human_subset
python scripts/compare_runs.py --base human_subset --new llm_distilled
```